In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
import re
import datetime as dt
import numpy as np
import pandas as pd

Mounted at /content/drive


In [2]:
PROJECT_DIR = "/content/drive/MyDrive/Insider-Market Trend Co-relation with ML"

INSIDER_PATH_RAW = "/content/drive/MyDrive/Insider-Market Trend Co-relation with ML /JPM Insider.xlsx"
PRICE_PATH_RAW   = "/content/drive/MyDrive/Insider-Market Trend Co-relation with ML /JPMorgan Stock Price History.xlsx"

def pick_existing_path(path_raw: str) -> str:
    """
    Tries the exact string you provided first.
    If not found, tries stripping accidental extra spaces (common copy/paste issue).
    """
    if os.path.exists(path_raw):
        return path_raw
    candidate = path_raw.strip()
    if os.path.exists(candidate):
        print("⚠️ Note: Your path had leading/trailing spaces. Using stripped path that exists.")
        return candidate
    # Also try removing ' /' to '/' (a very common Drive path typo)
    candidate2 = path_raw.replace(" /", "/")
    if os.path.exists(candidate2):
        print("⚠️ Note: Your path had ' /' before the filename. Using corrected path that exists.")
        return candidate2
    raise FileNotFoundError(f"File not found.\nTried:\n1) {path_raw}\n2) {candidate}\n3) {candidate2}")

INSIDER_PATH = pick_existing_path(INSIDER_PATH_RAW)
PRICE_PATH   = pick_existing_path(PRICE_PATH_RAW)

print("Insider file:", INSIDER_PATH)
print("Price file:  ", PRICE_PATH)
print("Project dir:", PROJECT_DIR)

Insider file: /content/drive/MyDrive/Insider-Market Trend Co-relation with ML /JPM Insider.xlsx
Price file:   /content/drive/MyDrive/Insider-Market Trend Co-relation with ML /JPMorgan Stock Price History.xlsx
Project dir: /content/drive/MyDrive/Insider-Market Trend Co-relation with ML


In [3]:
# Read raw data
ins_raw = pd.read_excel(INSIDER_PATH)
px_raw  = pd.read_excel(PRICE_PATH)

print("Insider raw shape:", ins_raw.shape)
print("Price raw shape:  ", px_raw.shape)

display(ins_raw.head(5))
display(px_raw.head(5))

Insider raw shape: (1001, 12)
Price raw shape:   (2805, 7)


,Unnamed: 0,Filing Date,Trade Date,\t\nTicker,Insider Name,Insider Title,Trade Type,Price,Qty,Owned,Δown%,Value
0,X,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2026-02-19 17:36:59,2026-02-19,JPM,Leopold Robin,Head of HR,S,$307.14,-432.0,83755.0,-0.01,"-$132,684"
2,NaN,2026-02-19 17:33:57,2026-02-19,JPM,Rohrbaugh Troy L,Co-CEO CIB,S,$307.11,-50000.0,111371.0,-0.31,"-$15,355,670"
3,NaN,2026-02-19 17:19:53,2026-02-19,JPM,Dimon James,"COB, CEO",S,$307.19,-69512.0,6275645.0,-0.01,"-$21,353,561"
4,NaN,2026-02-17 17:21:20,2026-02-17,JPM,Petno Douglas B,Co-CEO CIB,S,$306.40,-3487.0,435285.0,-0.01,"-$1,068,429"


,Date,Price,Open,High,Low,Vol.,Change %
0,02/27/2026,300.30,300.00,302.95,294.45,18.62M,-0.0190
1,02/26/2026,306.13,304.58,309.01,303.64,7.01M,0.0093
2,02/25/2026,303.30,298.64,303.66,297.01,8.10M,0.0202
3,02/24/2026,297.30,296.82,299.75,291.38,13.55M,-0.0012
4,02/23/2026,297.67,308.80,311.00,295.10,12.96M,-0.0422


In [4]:
print("=== INSIDER RAW INFO ===")
display(ins_raw.info())

print("\n=== PRICE RAW INFO ===")
display(px_raw.info())

print("\n=== INSIDER RAW COLUMNS ===")
print(list(ins_raw.columns))

print("\n=== PRICE RAW COLUMNS ===")
print(list(px_raw.columns))

=== INSIDER RAW INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Unnamed: 0     373 non-null    object        
 1   Filing Date    1000 non-null   datetime64[ns]
 2   Trade Date     1000 non-null   datetime64[ns]
 3   	
Ticker       1000 non-null   object        
 4   Insider Name   1000 non-null   object        
 5   Insider Title  1000 non-null   object        
 6   Trade Type     1000 non-null   object        
 7   Price          1000 non-null   object        
 8   Qty            1000 non-null   float64       
 9   Owned          1000 non-null   float64       
 10  Δown%          1000 non-null   object        
 11  Value          1000 non-null   object        
dtypes: datetime64[ns](2), float64(2), object(8)
memory usage: 94.0+ KB


None


=== PRICE RAW INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2805 entries, 0 to 2804
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      2805 non-null   object 
 1   Price     2805 non-null   float64
 2   Open      2805 non-null   float64
 3   High      2805 non-null   float64
 4   Low       2805 non-null   float64
 5   Vol.      2805 non-null   object 
 6   Change %  2805 non-null   float64
dtypes: float64(5), object(2)
memory usage: 153.5+ KB


None


=== INSIDER RAW COLUMNS ===
['Unnamed: 0', 'Filing\xa0Date', 'Trade Date', '\t\nTicker', 'Insider Name', 'Insider Title', 'Trade\xa0Type\xa0\xa0', 'Price', 'Qty', 'Owned', 'Δown%', 'Value']

=== PRICE RAW COLUMNS ===
['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']


In [5]:
def clean_col(c: str) -> str:
    # remove weird whitespace (NBSP, tabs, newlines) and normalize to snake_case
    c = str(c).replace("\xa0", " ").replace("\t", " ").replace("\n", " ").strip()
    c = re.sub(r"\s+", "_", c)
    return c.strip("_").lower()

def parse_money(x):
    """Parses values like '$307.14', '-$1,068,429' into floats."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s[1:-1]

    s = s.replace("$", "").replace(",", "").replace(" ", "")
    if s.startswith("-"):
        neg = True
        s = s[1:]
    elif s.startswith("+"):
        s = s[1:]

    try:
        v = float(s)
        return -v if neg else v
    except:
        return np.nan

def parse_percent(x):
    """Parses values like '-0.31' (meaning -0.31%) into decimal (-0.0031)."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace("%", "")
    if s == "" or s.lower() == "nan":
        return np.nan
    try:
        return float(s) / 100.0
    except:
        return np.nan

def parse_volume(v):
    """Parses '18.62M' into 18620000.0 etc."""
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.number)):
        return float(v)

    s = str(v).strip().replace(",", "")
    mult = 1.0
    if s.endswith("K"):
        mult = 1e3; s = s[:-1]
    elif s.endswith("M"):
        mult = 1e6; s = s[:-1]
    elif s.endswith("B"):
        mult = 1e9; s = s[:-1]

    try:
        return float(s) * mult
    except:
        return np.nan

def correct_excel_date_mess(x):
    """
    Your price file has mixed date types:
    - strings like '02/27/2026' => must be mm/dd/yyyy (month 27 impossible)
    - Excel-parsed datetimes that are often swapped (day/month)
    This function fixes that robustly.
    """
    if pd.isna(x):
        return pd.NaT

    # String dates in your file are mm/dd/yyyy
    if isinstance(x, str):
        return pd.to_datetime(x.strip(), format="%m/%d/%Y", errors="coerce")

    # If Excel already converted it to datetime, swap month/day
    if isinstance(x, (dt.datetime, dt.date, np.datetime64, pd.Timestamp)):
        ts = pd.to_datetime(x)
        return pd.Timestamp(year=ts.year, month=ts.day, day=ts.month)

    return pd.to_datetime(x, errors="coerce")

In [6]:
px = px_raw.copy()

# Standardize column names
px.columns = [clean_col(c) for c in px.columns]

# Expected columns check
expected_px_cols = {"date", "price", "open", "high", "low", "vol.", "change_%"}
# Sometimes "vol." becomes "vol." or "vol" depending on Excel; we'll handle both.
print("Price columns:", px.columns.tolist())

# Fix date
px["date"] = px["date"].apply(correct_excel_date_mess)

# Parse volume (handles 'Vol.' -> likely 'vol.' after clean_col)
vol_col = None
for candidate in ["vol.", "vol"]:
    if candidate in px.columns:
        vol_col = candidate
        break
if vol_col is None:
    raise KeyError("Could not find volume column after cleaning (expected something like 'vol.' or 'vol').")

px["volume"] = px[vol_col].apply(parse_volume)

# Sort chronologically (important for time-series work)
px = px.sort_values("date").reset_index(drop=True)

# Add dd/mm/yyyy display column (your preference)
px["date_ddmmyyyy"] = px["date"].dt.strftime("%d/%m/%Y")

display(px.head(5))
display(px.tail(5))

print("Price cleaned shape:", px.shape)
print("Price date range:", px["date"].min(), "→", px["date"].max())

Price columns: ['date', 'price', 'open', 'high', 'low', 'vol.', 'change_%']


,date,price,open,high,low,vol.,change_%,volume,date_ddmmyyyy
0,2015-01-02,62.49,62.62,62.96,62.07,12.60M,-0.0014,12600000.0,02/01/2015
1,2015-01-05,60.55,62.06,62.28,60.23,20.10M,-0.0310,20100000.0,05/01/2015
2,2015-01-06,58.98,60.64,60.75,58.35,29.07M,-0.0259,29070000.0,06/01/2015
3,2015-01-07,59.07,59.89,59.89,58.66,23.84M,0.0015,23840000.0,07/01/2015
4,2015-01-08,60.39,59.97,60.90,59.97,16.97M,0.0223,16970000.0,08/01/2015


,date,price,open,high,low,vol.,change_%,volume,date_ddmmyyyy
2800,2026-02-23,297.67,308.80,311.00,295.10,12.96M,-0.0422,12960000.0,23/02/2026
2801,2026-02-24,297.30,296.82,299.75,291.38,13.55M,-0.0012,13550000.0,24/02/2026
2802,2026-02-25,303.30,298.64,303.66,297.01,8.10M,0.0202,8100000.0,25/02/2026
2803,2026-02-26,306.13,304.58,309.01,303.64,7.01M,0.0093,7010000.0,26/02/2026
2804,2026-02-27,300.30,300.00,302.95,294.45,18.62M,-0.0190,18620000.0,27/02/2026


Price cleaned shape: (2805, 9)
Price date range: 2015-01-02 00:00:00 → 2026-02-27 00:00:00


In [7]:
# Weekend check (should be 0 for true trading days)
weekend_count = (px["date"].dt.weekday >= 5).sum()
print("Weekend rows (should be 0):", weekend_count)

# Missingness
print("\n=== PRICE MISSINGNESS (fraction) ===")
display(px.isna().mean().sort_values(ascending=False))

# Duplicate rows
dup_px = px.duplicated(subset=["date", "price", "open", "high", "low"]).sum()
print("\nDuplicate price rows:", dup_px)

# Sanity: reported % change vs computed close-to-close
px["ret_close"] = px["price"].pct_change()
# change_% is already numeric in your file; if it isn't, you can parse it later
if "change_%" in px.columns:
    px["ret_reported"] = px["change_%"]
    diff = (px["ret_close"] - px["ret_reported"]).abs()
    print("\nReturn sanity check |abs(computed - reported)| summary:")
    display(diff.describe())

Weekend rows (should be 0): 0

=== PRICE MISSINGNESS (fraction) ===


,0
date,0.0
price,0.0
open,0.0
high,0.0
low,0.0
vol.,0.0
change_%,0.0
volume,0.0
date_ddmmyyyy,0.0



Duplicate price rows: 0

Return sanity check |abs(computed - reported)| summary:


,0
count,2804.000000
mean,0.000025
std,0.000014
min,0.000000
25%,0.000013
50%,0.000026
75%,0.000038
max,0.000050


In [8]:
ins = ins_raw.copy()

# Standardize column names
ins.columns = [clean_col(c) for c in ins.columns]
print("Insider columns:", ins.columns.tolist())

# Drop obvious junk header row(s): your file has a first row where filing date is NaT
if "filing_date" not in ins.columns:
    raise KeyError("Expected a 'filing_date' column after cleaning. Check the insider headers.")

ins = ins[ins["filing_date"].notna()].copy()

# Parse dates safely (Excel usually loads them as datetime already)
ins["filing_date"] = pd.to_datetime(ins["filing_date"], errors="coerce")
ins["trade_date"]  = pd.to_datetime(ins["trade_date"], errors="coerce")

# Parse numeric columns
ins["trade_price"] = ins["price"].apply(parse_money)
ins["trade_value"] = ins["value"].apply(parse_money)

# Δown% may be in a column name like 'δown%' after cleaning
delta_col = None
for c in ins.columns:
    if "own" in c and "%" in c:
        delta_col = c
        break
if delta_col is None:
    # fallback to common cleaned name
    if "δown%" in ins.columns:
        delta_col = "δown%"
if delta_col is not None:
    ins["delta_own_pct"] = ins[delta_col].apply(parse_percent)
else:
    ins["delta_own_pct"] = np.nan

# Create dd/mm/yyyy display columns
ins["trade_date_ddmmyyyy"]  = ins["trade_date"].dt.strftime("%d/%m/%Y")
ins["filing_date_ddmmyyyy"] = ins["filing_date"].dt.strftime("%d/%m/%Y")

display(ins.head(5))
print("Insider cleaned shape:", ins.shape)
print(" Trade date range:", ins["trade_date"].min(), "→", ins["trade_date"].max())
print(" Filing date range:", ins["filing_date"].min(), "→", ins["filing_date"].max())

Insider columns: ['unnamed:_0', 'filing_date', 'trade_date', 'ticker', 'insider_name', 'insider_title', 'trade_type', 'price', 'qty', 'owned', 'δown%', 'value']


,unnamed:_0,filing_date,trade_date,ticker,insider_name,insider_title,trade_type,price,qty,owned,δown%,value,trade_price,trade_value,delta_own_pct,trade_date_ddmmyyyy,filing_date_ddmmyyyy
1,NaN,2026-02-19 17:36:59,2026-02-19,JPM,Leopold Robin,Head of HR,S,$307.14,-432.0,83755.0,-0.01,"-$132,684",307.14,-132684.0,-0.0001,19/02/2026,19/02/2026
2,NaN,2026-02-19 17:33:57,2026-02-19,JPM,Rohrbaugh Troy L,Co-CEO CIB,S,$307.11,-50000.0,111371.0,-0.31,"-$15,355,670",307.11,-15355670.0,-0.0031,19/02/2026,19/02/2026
3,NaN,2026-02-19 17:19:53,2026-02-19,JPM,Dimon James,"COB, CEO",S,$307.19,-69512.0,6275645.0,-0.01,"-$21,353,561",307.19,-21353561.0,-0.0001,19/02/2026,19/02/2026
4,NaN,2026-02-17 17:21:20,2026-02-17,JPM,Petno Douglas B,Co-CEO CIB,S,$306.40,-3487.0,435285.0,-0.01,"-$1,068,429",306.40,-1068429.0,-0.0001,17/02/2026,17/02/2026
5,NaN,2026-02-17 17:21:19,2026-02-17,JPM,Erdoes Mary E.,"CEO Asset, Wealth Management",S,$306.41,-5731.0,613405.0,-0.01,"-$1,756,046",306.41,-1756046.0,-0.0001,17/02/2026,17/02/2026


Insider cleaned shape: (1000, 17)
 Trade date range: 2014-11-14 00:00:00 → 2026-02-19 00:00:00
 Filing date range: 2015-01-05 17:27:14 → 2026-02-19 17:36:59


In [9]:
print("=== INSIDER MISSINGNESS (fraction) ===")
display(ins.isna().mean().sort_values(ascending=False))

# Duplicates (strict)
dup_ins = ins.duplicated(subset=[
    "filing_date","trade_date","ticker","insider_name","insider_title",
    "trade_type","trade_price","qty","owned","trade_value"
]).sum()
print("Duplicate insider rows:", dup_ins)

# Trade types breakdown
print("\n=== TRADE TYPE COUNTS ===")
display(ins["trade_type"].value_counts())

# Value consistency: trade_value ≈ qty * trade_price
ins["calc_value"] = ins["qty"] * ins["trade_price"]
ins["value_abs_diff"] = (ins["trade_value"] - ins["calc_value"]).abs()

print("\nValue consistency |trade_value - qty*price| summary:")
display(ins["value_abs_diff"].describe())

print("\nTop 10 largest mismatches (inspect):")
display(
    ins.sort_values("value_abs_diff", ascending=False)
      .head(10)[["filing_date","trade_date","insider_name","trade_type","trade_price","qty","trade_value","calc_value","value_abs_diff"]]
)

=== INSIDER MISSINGNESS (fraction) ===


,0
unnamed:_0,0.628
delta_own_pct,0.005
filing_date,0.000
ticker,0.000
trade_date,0.000
insider_title,0.000
trade_type,0.000
price,0.000
insider_name,0.000
qty,0.000


Duplicate insider rows: 0

=== TRADE TYPE COUNTS ===


,count
trade_type,
A,406
F,288
S,158
S+OE,67
G,58
P,22
D,1



Value consistency |trade_value - qty*price| summary:


,value_abs_diff
count,1000.000000
mean,75.333800
std,325.292262
min,0.000000
25%,5.580000
50%,23.145000
75%,49.455000
max,7418.560000



Top 10 largest mismatches (inspect):


,filing_date,trade_date,insider_name,trade_type,trade_price,qty,trade_value,calc_value,value_abs_diff
749,2017-10-13 17:12:09,2017-10-13,Dimon James,F,95.84,-1483634.0,-142184064.0,-1.421915e+08,7418.56
189,2024-02-22 17:51:39,2024-02-22,Dimon James,S,182.74,-821778.0,-150167221.0,-1.501717e+08,4490.72
981,2015-01-16 16:49:48,2015-01-15,Dimon James,F,55.56,-514120.0,-28561937.0,-2.856451e+07,2570.20
100,2025-02-20 18:21:49,2025-02-20,Dimon James,S,269.84,-866361.0,-233776513.0,-2.337789e+08,2339.24
863,2016-04-20 17:23:03,2016-04-19,Smith Gordon,F,63.02,-452531.0,-28516241.0,-2.851850e+07,2262.62
405,2021-10-15 16:16:10,2021-10-14,Dimon James,F,162.04,-375403.0,-60828425.0,-6.083030e+07,1877.12
594,2019-07-24 16:56:30,2019-07-23,Smith Gordon,F,115.64,-231700.0,-26792630.0,-2.679379e+07,1158.00
848,2016-08-12 16:04:09,2016-08-11,Erdoes Mary E.,F,65.36,-228347.0,-14923618.0,-1.492476e+07,1141.92
521,2020-03-27 16:17:27,2020-03-25,Dimon James,F,89.85,-219392.0,-19711244.0,-1.971237e+07,1127.20
950,2015-05-22 15:25:38,2015-05-21,Erdoes Mary E.,F,66.35,-173957.0,-11541177.0,-1.154205e+07,869.95


In [10]:
# Trading calendar from price file
trading_days = set(px["date"].dt.normalize())

ins_trade_days = set(ins["trade_date"].dt.normalize())
missing_trade_days = sorted(list(ins_trade_days - trading_days))

print("Insider trade dates not present in price trading calendar:", len(missing_trade_days))
print("Sample missing trade dates (first 20):", missing_trade_days[:20])

# This is normal for:
# - trades before 2015-01-02 (your price file starts 2015)
# - trades on weekends/holidays

Insider trade dates not present in price trading calendar: 23
Sample missing trade dates (first 20): [Timestamp('2014-11-14 00:00:00'), Timestamp('2014-12-31 00:00:00'), Timestamp('2015-07-25 00:00:00'), Timestamp('2015-10-25 00:00:00'), Timestamp('2016-12-31 00:00:00'), Timestamp('2017-09-30 00:00:00'), Timestamp('2017-12-31 00:00:00'), Timestamp('2018-01-13 00:00:00'), Timestamp('2018-03-31 00:00:00'), Timestamp('2018-06-30 00:00:00'), Timestamp('2018-09-30 00:00:00'), Timestamp('2019-01-13 00:00:00'), Timestamp('2019-03-31 00:00:00'), Timestamp('2019-06-30 00:00:00'), Timestamp('2020-10-25 00:00:00'), Timestamp('2022-12-31 00:00:00'), Timestamp('2023-03-25 00:00:00'), Timestamp('2023-09-30 00:00:00'), Timestamp('2023-12-31 00:00:00'), Timestamp('2024-01-13 00:00:00')]


In [11]:
OUT_DIR = os.path.join(PROJECT_DIR, "processed")
os.makedirs(OUT_DIR, exist_ok=True)

px_out = os.path.join(OUT_DIR, "jpm_prices_clean.csv")
ins_out = os.path.join(OUT_DIR, "jpm_insider_clean.csv")

px.to_csv(px_out, index=False)
ins.to_csv(ins_out, index=False)

print(" Saved:", px_out)
print(" Saved:", ins_out)

 Saved: /content/drive/MyDrive/Insider-Market Trend Co-relation with ML/processed/jpm_prices_clean.csv
 Saved: /content/drive/MyDrive/Insider-Market Trend Co-relation with ML/processed/jpm_insider_clean.csv


In [12]:
summary = {
    "PRICE_rows": len(px),
    "PRICE_date_min": str(px["date"].min().date()),
    "PRICE_date_max": str(px["date"].max().date()),
    "PRICE_weekend_rows": int((px["date"].dt.weekday >= 5).sum()),
    "INS_rows": len(ins),
    "INS_trade_date_min": str(ins["trade_date"].min().date()),
    "INS_trade_date_max": str(ins["trade_date"].max().date()),
    "INS_unique_insiders": int(ins["insider_name"].nunique()),
    "INS_trade_type_counts": ins["trade_type"].value_counts().to_dict(),
    "INS_value_mismatch_median_$": float(ins["value_abs_diff"].median()),
}

pd.DataFrame([summary]).T.rename(columns={0:"value"})

,value
PRICE_rows,2805
PRICE_date_min,2015-01-02
PRICE_date_max,2026-02-27
PRICE_weekend_rows,0
INS_rows,1000
INS_trade_date_min,2014-11-14
INS_trade_date_max,2026-02-19
INS_unique_insiders,40
INS_trade_type_counts,"{'A': 406, 'F': 288, 'S': 158, 'S+OE': 67, 'G'..."
INS_value_mismatch_median_$,23.145


In [13]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Insider-Market Trend Co-relation with ML /cleaned & processed"

CLEANED_DIR = os.path.join(PROJECT_DIR, "cleaned")
PROCESSED_DIR = os.path.join(PROJECT_DIR, "processed")  # optional compatibility

os.makedirs(CLEANED_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

px_clean_path = os.path.join(CLEANED_DIR, "jpm_prices_clean.csv")
ins_clean_path = os.path.join(CLEANED_DIR, "jpm_insider_clean.csv")

px.to_csv(px_clean_path, index=False)
ins.to_csv(ins_clean_path, index=False)

# Optional: also save to /processed/ so future notebooks can use either convention
px.to_csv(os.path.join(PROCESSED_DIR, "jpm_prices_clean.csv"), index=False)
ins.to_csv(os.path.join(PROCESSED_DIR, "jpm_insider_clean.csv"), index=False)

print("Saved cleaned price CSV :", px_clean_path)
print("Saved cleaned insider CSV:", ins_clean_path)
print("Also saved copies into   :", PROCESSED_DIR)

Saved cleaned price CSV : /content/drive/MyDrive/Insider-Market Trend Co-relation with ML /cleaned & processed/cleaned/jpm_prices_clean.csv
Saved cleaned insider CSV: /content/drive/MyDrive/Insider-Market Trend Co-relation with ML /cleaned & processed/cleaned/jpm_insider_clean.csv
Also saved copies into   : /content/drive/MyDrive/Insider-Market Trend Co-relation with ML /cleaned & processed/processed
